# 98 Degradation CE / ASR (EP FIGS. 19-27 analog)

6 M KOH, 303 K, isolation protects ORR.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path
setup_path()
from plant_sim.params import default_params, CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G
from plant_sim.plotting import apply_style, timeseries, xy_plot, FIGURES
apply_style()
P = default_params()
print("6 M KOH default", P.c_KOH_mol_m3/1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G,2), round(CAPACITY_MAGNETITE_MAH_G,2))
print("figures ->", FIGURES)


In [ ]:
from plant_sim.components.degradation import Degradation
from plant_sim.components.base import integrate_component
deg = Degradation(P)
t, y = integrate_component(lambda t,y: deg.rhs(t,y,{"I_cell_A": 2.0, "eta_oer_V": 0.55, "r_carb_mol_s": 1e-7, "saturation": 0.25, "oer_isolated": 0.0 if t>8e4 else 1.0}, {}), deg.y0(), (0, 2e5), n_eval=200)
fade = [deg.outputs(float(ti), y[:,i], {}, {})["CE_fade"] for i,ti in enumerate(t)]
timeseries(t/86400, {"CE_fade": fade, "ASR": y[1], "ORR_loss": y[2], "PTFE": y[3]}, xlabel="t (day)", ylabel="fade metrics", title="CE/ASR aging analog of EP FIGS. 19-27", callout="6 M KOH, 303 K", stem="nb98")
